# 07 Final Results, Reproducibility, And Limitations

Final professor-facing checklist tying assignment deliverables, validation commands, and visible limitations together.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Final Artifact Snapshot


In [2]:
{
    "required_assignment_outputs": {
        "measures.csv": csv_count("outputs/measures.csv"),
        "dimension_names.csv": csv_count("outputs/dimension_names.csv"),
        "dimension_values.csv": csv_count("outputs/dimension_values.csv"),
        "units.csv": csv_count("outputs/units.csv"),
    },
    "supplementary_outputs": {
        "other_ambiguous.csv": csv_count("outputs/other_ambiguous.csv"),
        "measure_clusters.csv": csv_count("outputs/measure_clusters.csv"),
        "measure_relations.csv": csv_count("outputs/measure_relations.csv"),
    },
    "classification_run": "run_75b90575bb9e48ce7918",
    "cluster_run": read_json("report/clustering_metrics.json")
    .get("latest_run_summary", {})
    .get("run_id"),
    "relation_run": read_json("report/relations_metrics.json")
    .get("latest_run_summary", {})
    .get("run_id"),
    "search_index_run": read_json("report/retrieval_metrics.json")
    .get("index", {})
    .get("run_id"),
}

{'required_assignment_outputs': {'measures.csv': 2894,
  'dimension_names.csv': 359,
  'dimension_values.csv': 607,
  'units.csv': 261},
 'supplementary_outputs': {'other_ambiguous.csv': 9114,
  'measure_clusters.csv': 2894,
  'measure_relations.csv': 30695},
 'classification_run': 'run_75b90575bb9e48ce7918',
 'cluster_run': 'run_e1c656105fd6bcd3c088',
 'relation_run': 'run_9431e98d07745bcef11d',
 'search_index_run': 'run_391c6ea54050920593d9'}

## Validation Commands


In [3]:
[
    "pytest tests/test_notebooks.py",
    "statvocab validate-artifacts --config configs/core.yaml --run-id run_75b90575bb9e48ce7918",
    (
        "pytest tests/test_classification_contracts.py tests/test_grounding.py "
        "tests/test_relations.py tests/test_notebooks.py"
    ),
]

['pytest tests/test_notebooks.py',
 'statvocab validate-artifacts --config configs/core.yaml --run-id run_75b90575bb9e48ce7918',
 'pytest tests/test_classification_contracts.py tests/test_grounding.py tests/test_relations.py tests/test_notebooks.py']

## Visible Limitations


In [4]:
[
    (
        "Step-6 audit labels were completed with a rule-assisted repository audit; "
        "duplicate agreement is available but not a fully independent second-human study."
    ),
    (
        "The local classifier under-recovers some measure labels and moves uncertain "
        "measure-like terms to other_ambiguous."
    ),
    (
        "The refreshed clustering manual review sample is complete, but it still "
        "shows a large unclustered bucket and domain-label misses."
    ),
    (
        "The refreshed relationship manual review sample is complete, but broad "
        "related_to edges and lexical-containment false positives remain."
    ),
    (
        "The 7,605-table scale experiment has a completed ingestion checkpoint; "
        "extraction and downstream scale results remain pending."
    ),
]

['Step-6 audit labels were completed with a rule-assisted repository audit; duplicate agreement is available but not a fully independent second-human study.',
 'The local classifier under-recovers some measure labels and moves uncertain measure-like terms to other_ambiguous.',
 'The refreshed clustering manual review sample is complete, but it still shows a large unclustered bucket and domain-label misses.',
 'The refreshed relationship manual review sample is complete, but broad related_to edges and lexical-containment false positives remain.',
 'The 7,605-table scale experiment has a completed ingestion checkpoint; extraction and downstream scale results remain pending.']